In [ ]:
import os
import numpy as np
from PIL import Image
import tifffile
from pathlib import Path

def stack_16bit_tiffs_simple(folder_path, output_base_folder=None):
    """
    Simple stacking version specifically for 16-bit TIFF files.
    TIF and JPG are saved in separate folders, no subfolders created.
    
    Args:
        folder_path: Path to the main folder containing subfolders
        output_base_folder: Path to the output base folder
    """
    
    # Set output folder
    if output_base_folder is None:
        output_base_folder = os.path.join(os.path.dirname(folder_path), 'DLBCL RGB normalized')
    
    # Create two separate output folders for TIF and JPG
    tif_output_folder = os.path.join(output_base_folder, 'tif_files')
    jpg_output_folder = os.path.join(output_base_folder, 'jpg_files')
    
    os.makedirs(tif_output_folder, exist_ok=True)
    os.makedirs(jpg_output_folder, exist_ok=True)
    
    print(f"TIF files will be saved to: {tif_output_folder}")
    print(f"JPG files will be saved to: {jpg_output_folder}")
    
    # Get all subfolders
    subfolders = [f for f in os.listdir(folder_path) 
                 if os.path.isdir(os.path.join(folder_path, f))]
    
    print(f"Found {len(subfolders)} subfolders")
    
    total_processed = 0
    total_skipped = 0
    
    # Iterate through all subfolders
    for subfolder_name in subfolders:
        subfolder_path = os.path.join(folder_path, subfolder_name)
        
        print(f"\nProcessing subfolder: {subfolder_name}")
        
        # Check if all three channel folders exist
        r_folder = os.path.join(subfolder_path, 'formatted_ccr7', 'normalized_tif')
        g_folder = os.path.join(subfolder_path, 'formatted_actin', 'normalized_tif')
        b_folder = os.path.join(subfolder_path, 'formatted_cd45ra', 'normalized_tif')
        
        if not all(os.path.exists(f) for f in [r_folder, g_folder, b_folder]):
            print(f"  Skipping: missing required channel folders")
            continue
        
        # Get all tif files from R channel folder
        try:
            r_images = [f for f in os.listdir(r_folder) if f.endswith('.tif')]
        except:
            print(f"  Cannot read folder: {r_folder}")
            continue
        
        if not r_images:
            print(f"  No tif files found in {r_folder}")
            continue
        
        processed = 0
        skipped = 0
        
        for r_image in r_images:
            try:
                # Extract base filename (remove _ccr7_normalized.tif)
                if '_ccr7_normalized.tif' in r_image:
                    base_name = r_image.replace('_ccr7_normalized.tif','')
                else:
                    base_name = r_image.replace('.tif','')
                
                # Build corresponding G and B channel filenames
                g_image = base_name + '_actin_normalized.tif'
                b_image = base_name + '_cd45ra_normalized.tif'
                
                # Full file paths
                r_path = os.path.join(r_folder, r_image)
                g_path = os.path.join(g_folder, g_image)
                b_path = os.path.join(b_folder, b_image)
                
                # Check if files exist
                if not os.path.exists(g_path):
                    print(f"  G channel not found: {g_path}")
                    skipped += 1
                    continue
                if not os.path.exists(b_path):
                    print(f"  B channel not found: {b_path}")
                    skipped += 1
                    continue
                
                # Read 16-bit TIFF using tifffile
                print(f"  Reading: {base_name}")
                r_array = tifffile.imread(r_path)
                g_array = tifffile.imread(g_path)
                b_array = tifffile.imread(b_path)
                
                # Check data type
                print(f"    R channel: shape={r_array.shape}, dtype={r_array.dtype}, range=[{r_array.min()}, {r_array.max()}]")
                print(f"    G channel: shape={g_array.shape}, dtype={g_array.dtype}, range=[{g_array.min()}, {g_array.max()}]")
                print(f"    B channel: shape={b_array.shape}, dtype={b_array.dtype}, range=[{b_array.min()}, {b_array.max()}]")
                
                # Ensure 2D array (if 3D with single channel, drop channel dimension)
                if len(r_array.shape) == 3 and r_array.shape[2] == 1:
                    r_array = r_array[:, :, 0]
                if len(g_array.shape) == 3 and g_array.shape[2] == 1:
                    g_array = g_array[:, :, 0]
                if len(b_array.shape) == 3 and b_array.shape[2] == 1:
                    b_array = b_array[:, :, 0]
                
                # Check that dimensions match
                if r_array.shape != g_array.shape or r_array.shape != b_array.shape:
                    print(f"  Error: image dimensions do not match")
                    print(f"    R: {r_array.shape}, G: {g_array.shape}, B: {b_array.shape}")
                    skipped += 1
                    continue
                
                # Create RGB array (keep 16-bit)
                height, width = r_array.shape
                rgb_array = np.zeros((height, width, 3), dtype=np.float32)
                
                # Assign channels
                rgb_array[:, :, 0] = r_array  # R channel
                rgb_array[:, :, 1] = g_array  # G channel
                rgb_array[:, :, 2] = b_array  # B channel
                
                # Generate output filename
                output_name = base_name + '_rgb'
                
                # Save 16-bit TIFF to tif_files folder
                tif_path = os.path.join(tif_output_folder, output_name + '.tif')
                tifffile.imwrite(tif_path, rgb_array, 
                               photometric='rgb',
                               compression=None,
                               metadata={'axes': 'YXC'})
                
                # Create 8-bit version for JPG saving
                # Method 1: linear scaling (if data range is appropriate)
                #if rgb_array.max() > 255:
                    # Linear scale to 0-255 range
                 #   rgb_8bit = ((rgb_array - rgb_array.min()) * 255.0 / (rgb_array.max() - rgb_array.min())).astype(np.uint8)
                #else:
                    # Direct conversion
                rgb_8bit = rgb_array.astype(np.uint32)
                
                # Save JPG to jpg_files folder
                jpg_path = os.path.join(jpg_output_folder, output_name + '.jpg')
                Image.fromarray(rgb_8bit).save(jpg_path, quality=95, optimize=True)
                
                print(f"  Saved: {output_name}.tif (16-bit) to tif_files/")
                print(f"         {output_name}.jpg (8-bit preview) to jpg_files/")
                processed += 1
                
            except Exception as e:
                print(f"  Error processing {r_image}: {str(e)}")
                import traceback
                traceback.print_exc()
                skipped += 1
                continue
        
        print(f"  Subfolder stats: processed {processed} image sets, skipped {skipped} sets")
        total_processed += processed
        total_skipped += skipped
    
    print("\n" + "="*60)
    print(f"All processing complete!")
    print(f"Total processed: {total_processed} image sets")
    print(f"Total skipped: {total_skipped} image sets")
    print(f"TIF files saved to: {tif_output_folder}")
    print(f"JPG files saved to: {jpg_output_folder}")
    print("="*60)

def main():
    # Update this path to your main folder path
    main_folder = r"/mnt/HDD16TB/LanceKam_Lab/Daizong/Project/DLBCL/DLBCL/DLBCL_processed"  # Modify to your path
    
    if not os.path.exists(main_folder):
        print(f"Error: folder not found {main_folder}")
        return
    
    print("="*60)
    print("16-bit TIFF image stacking")
    print("="*60)
    print(f"Input folder: {main_folder}")
    
    # Run conversion
    stack_16bit_tiffs_simple(main_folder)
    
    print("\nAll files processed!")

if __name__ == "__main__":
    # Make sure required libraries are installed
    # pip install tifffile numpy pillow
    main()